# Tutorial: GEN-01 Variational Autoencoder on Digits

- Module: 11 Generative Models
- Topic: variational autoencoders and the ELBO
- Estimated runtime: 2-3 minutes on CPU
- Prerequisites: latent-variable models, KL divergence, stochastic gradient descent, PyTorch basics
- Expected outputs: ELBO curves, reconstruction examples, and samples drawn from the learned latent space
    


## Learning goals

- Train a small VAE with a stochastic encoder and decoder.
- Separate the ELBO into reconstruction and KL terms.
- Use `sklearn.datasets.load_digits` as a local MNIST-like stand-in so the notebook remains offline and lightweight.
- Visualize reconstructions and samples from the latent prior.
    


In [ ]:
from __future__ import annotations

import math

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(7)
np.random.seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device
    


## Dataset and loader setup

The file name says `vae-mnist`, but to avoid network downloads we use the scikit-learn digits dataset.
Each image is `8 x 8` rather than `28 x 28`, yet it still supports the core VAE workflow: encode, sample, decode, and optimize the ELBO.
    


In [ ]:
digits = load_digits()
X = (digits.images / 16.0).astype(np.float32)
y = digits.target.astype(np.int64)
X = X.reshape(len(X), -1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)

train_tensor = torch.tensor(X_train)
test_tensor = torch.tensor(X_test)

train_loader = DataLoader(TensorDataset(train_tensor), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(test_tensor), batch_size=128, shuffle=False)

X_train.shape, X_test.shape
    


## VAE model

The encoder outputs the mean and log-variance of a diagonal Gaussian `q_\phi(z | x)`.
The decoder returns Bernoulli logits for each pixel, and the reparameterization trick writes

$$
z = \mu_\phi(x) + \sigma_\phi(x) \odot 
arepsilon, \qquad 
arepsilon \sim \mathcal{N}(0, I).
$$
    


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim: int = 64, hidden_dim: int = 128, latent_dim: int = 2) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.encoder(x)
        return self.mu_head(h), self.logvar_head(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


def vae_loss(x: torch.Tensor, logits: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor):
    reconstruction = nn.functional.binary_cross_entropy_with_logits(
        logits, x, reduction='sum'
    )
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total = reconstruction + kl
    return total, reconstruction, kl


model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model
    


## Training loop

We optimize the negative ELBO.
The reconstruction term rewards accurate decoded pixels, while the KL term keeps the approximate posterior close to the unit Gaussian prior.
    


In [ ]:
def run_epoch(loader: DataLoader, train: bool) -> dict[str, float]:
    model.train(train)
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    total_examples = 0

    for (xb,) in loader:
        xb = xb.to(device)
        logits, mu, logvar = model(xb)
        loss, recon, kl = vae_loss(xb, logits, mu, logvar)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = xb.size(0)
        total_examples += batch_size
        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    return {
        'loss_per_example': total_loss / total_examples,
        'recon_per_example': total_recon / total_examples,
        'kl_per_example': total_kl / total_examples,
    }


history = []
for epoch in range(1, 9):
    train_metrics = run_epoch(train_loader, train=True)
    test_metrics = run_epoch(test_loader, train=False)
    history.append({
        'epoch': epoch,
        'train_loss': train_metrics['loss_per_example'],
        'train_recon': train_metrics['recon_per_example'],
        'train_kl': train_metrics['kl_per_example'],
        'test_loss': test_metrics['loss_per_example'],
    })
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_metrics['loss_per_example']:.3f} "
        f"recon={train_metrics['recon_per_example']:.3f} "
        f"kl={train_metrics['kl_per_example']:.3f} "
        f"test_loss={test_metrics['loss_per_example']:.3f}"
    )

history[-1]
    


## ELBO diagnostics

A healthy run usually shows the reconstruction term dropping quickly and the KL term remaining much smaller but nonzero.
If the KL term collapses to nearly zero immediately, the decoder may be ignoring the latent code.
    


In [ ]:
epochs = [item['epoch'] for item in history]
train_loss = [item['train_loss'] for item in history]
train_recon = [item['train_recon'] for item in history]
train_kl = [item['train_kl'] for item in history]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(epochs, train_loss, marker='o', label='negative ELBO')
ax[0].plot(epochs, train_recon, marker='s', label='reconstruction')
ax[0].plot(epochs, train_kl, marker='^', label='KL')
ax[0].set_title('Training objective components')
ax[0].set_xlabel('Epoch')
ax[0].legend()

with torch.no_grad():
    mu, _ = model.encode(test_tensor[:300].to(device))
latent = mu.cpu().numpy()
scatter = ax[1].scatter(latent[:, 0], latent[:, 1], c=y_test[:300], cmap='tab10', s=16)
ax[1].set_title('Mean latent codes for test digits')
ax[1].set_xlabel('z1')
ax[1].set_ylabel('z2')
fig.colorbar(scatter, ax=ax[1], fraction=0.046, pad=0.04)
plt.tight_layout()
fig
    


## Reconstructions and prior samples

The first panel compares original digits with reconstructions.
The second panel decodes points sampled from the prior `p(z) = \mathcal{N}(0, I)`.
    


In [ ]:
model.eval()
with torch.no_grad():
    originals = test_tensor[:10].to(device)
    recon_logits, _, _ = model(originals)
    reconstructions = torch.sigmoid(recon_logits).cpu().numpy().reshape(-1, 8, 8)
    originals_np = originals.cpu().numpy().reshape(-1, 8, 8)
    prior_samples = torch.randn(10, 2, device=device)
    decoded = torch.sigmoid(model.decode(prior_samples)).cpu().numpy().reshape(-1, 8, 8)

fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for i in range(10):
    axes[0, i].imshow(originals_np[i], cmap='gray')
    axes[1, i].imshow(reconstructions[i], cmap='gray')
    axes[2, i].imshow(decoded[i], cmap='gray')
    for row in range(3):
        axes[row, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Recon', fontsize=10)
axes[2, 0].set_ylabel('Sample', fontsize=10)
plt.tight_layout()
fig
    


## Summary

This notebook makes the ELBO concrete: a VAE is a latent-variable model trained by stochastic optimization rather than exact EM.
The learned representation is smooth enough to sample from, but its likelihood-friendly objective can trade away visual sharpness compared with adversarial models.
    
